# Week 8 Lab Exercise — ARIA v5.0: The Matai'an Three-Act Auditor
# 第八週課堂練習 — ARIA v5.0：馬太鞍三幕稽核器

**Guided lab worksheet** — follow along with the instructor's Demo notebook and fill in the TODO cells.
**引導式練習** — 跟著老師的 Demo 操作，完成 TODO 標記的程式碼。

**Case / 案例**: 2025 Matai'an Creek Barrier Lake Event（馬太鞍溪堰塞湖事件）

---

### Class rhythm / 課程節奏

| Phase / 階段 | Content / 內容 | Cells | Time / 時間 |
|---|---|---|---|
| **Lab 1** | Spectral Detective 光譜偵探 | [S1]–[S6] | 35 min |
| **Lab 2** | Three-Act Audit 三幕稽核 | [S7]–[S15] | 50 min |

## Lab 1 — Environment + Three-Act STAC Scene Selection
## Lab 1 — 環境設定 + 三幕 STAC 影像選擇

In [ ]:
# [S1] Environment setup
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
from pyproj import Transformer

import pystac_client
import planetary_computer as pc
import stackstac
import rioxarray as rxr
import xarray as xr

# 中文字型設定
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# ── 連接 Planetary Computer STAC ──
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)
print("✅ 已連接 Planetary Computer STAC")

# ── MPC-friendly robust search ──
# MPC 有時在 bbox + datetime + server-side query 合用時吐出 504
# 解法：拿掉 server-side query，改用 client-side 過濾雲量
def robust_search(bbox, datetime_range, cloud_max=30, max_items=60, tries=3):
    """穩健的 STAC 搜尋：client-side 過濾雲量 + exponential back-off。"""
    for attempt in range(tries):
        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=bbox,
                datetime=datetime_range,
                max_items=max_items,
            )
            items = list(search.items())
            items = [i for i in items
                     if i.properties.get("eo:cloud_cover", 100) < cloud_max]
            items.sort(key=lambda i: i.properties["eo:cloud_cover"])
            return items
        except Exception as e:
            print(f"  ⚠️  搜尋嘗試 {attempt+1}/{tries} 失敗：{e}")
            time.sleep(2 ** attempt)
    raise RuntimeError("STAC search failed 3x")

# AOI — 馬太鞍溪集水區（上游萬榮崩塌源 → 下游光復鄉）
MATAIAN_BBOX = [121.28, 23.56, 121.52, 23.76]
WANTED_BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]

# 選定場景 ID（完成 S2–S4 TCI Quick-QA 後填入，確保可重現）
PRE_ITEM_ID  = "FILL_IN_AFTER_QA"
MID_ITEM_ID  = "FILL_IN_AFTER_QA"
POST_ITEM_ID = "FILL_IN_AFTER_QA"

In [ ]:
# [S2] ACT 1 — Pre-event STAC 搜尋（2025 年 6 月，颱風威帕前）
items_pre = robust_search(MATAIAN_BBOX, "2025-06-01/2025-07-15", cloud_max=20)
print(f"Pre-event (Jun–early Jul 2025)：找到 {len(items_pre)} 張低雲影像")
for item in items_pre[:5]:
    print(f"  {item.id}  |  clouds={item.properties['eo:cloud_cover']:.1f}%")

# ── TCI Quick-QA：視覺檢查前 3 個候選場景 ──
n = max(min(len(items_pre), 3), 1)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]

for ax, item in zip(axes, items_pre[:n]):
    tci = rxr.open_rasterio(item.assets["visual"].href, overview_level=3)
    rgb = np.moveaxis(tci.values[:3], 0, -1)   # (3,H,W) → (H,W,3)
    ax.imshow(rgb)
    ax.set_title(f"{item.id[:20]}\nclouds={item.properties['eo:cloud_cover']:.1f}%",
                 fontsize=8)
    ax.axis("off")

plt.suptitle("ACT 1 — Pre-event TCI Quick-QA", fontsize=12)
plt.tight_layout()
plt.show()

# 選雲量最低的場景
pre_item   = items_pre[0]
PRE_ITEM_ID = pre_item.id
print(f"\n✅ 選定 Pre 場景：{PRE_ITEM_ID}  "
      f"({pre_item.properties['eo:cloud_cover']:.1f}% clouds)")

### ✏️ Discussion / 討論（Act 1 — Pre）

**為什麼選這張事件前影像？**

選擇雲量最低（< 5%）且日期最接近颱風威帕（7 月 21 日）前的場景，確保馬太鞍谷地清晰可見，作為後續比對的最乾淨基準。6 月下旬至 7 月初是花蓮梅雨季尾聲，日照充足、雲量低，是台灣東部衛星影像品質最佳的窗口，TCI 可見完整的深綠色鬱閉森林——這正是事件前的正常狀態，沒有任何異常水體存在。

*(Why this Pre-event scene? Lowest cloud cover and closest date before Typhoon Wipha on July 21. Late June is Hualien's best pre-monsoon imaging window; TCI shows fully forested valley with no anomalous water body — a clean baseline for the three-act comparison.)*

In [ ]:
# [S3] ACT 2 — Mid-event STAC 搜尋（2025 年 8–9 月，堰塞湖已形成、尚未潰堤）
# 颱風季節放寬雲量至 40%
items_mid = robust_search(MATAIAN_BBOX, "2025-08-01/2025-09-20", cloud_max=40)
print(f"Mid-event (Aug–mid Sep 2025)：找到 {len(items_mid)} 張可用影像")
for item in items_mid[:5]:
    print(f"  {item.id}  |  clouds={item.properties['eo:cloud_cover']:.1f}%")

# ── TCI Quick-QA：尋找堰塞湖（應呈現青藍色水體）──
n = max(min(len(items_mid), 3), 1)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]

for ax, item in zip(axes, items_mid[:n]):
    tci = rxr.open_rasterio(item.assets["visual"].href, overview_level=3)
    rgb = np.moveaxis(tci.values[:3], 0, -1)
    ax.imshow(rgb)
    ax.set_title(f"{item.id[:20]}\nclouds={item.properties['eo:cloud_cover']:.1f}%",
                 fontsize=8)
    ax.axis("off")

plt.suptitle("ACT 2 — Mid-event TCI Quick-QA（尋找堰塞湖青色水體）", fontsize=12)
plt.tight_layout()
plt.show()

mid_item   = items_mid[0]
MID_ITEM_ID = mid_item.id
print(f"\n✅ 選定 Mid 場景：{MID_ITEM_ID}  "
      f"({mid_item.properties['eo:cloud_cover']:.1f}% clouds)")

### ✏️ Discussion / 討論（Act 2 — Mid）

**你找到了清楚顯示堰塞湖的影像嗎？為什麼選這一張？**

是的。在 8–9 月雲量偏高的影像中，挑選馬太鞍上游谷地（約 121.29°E, 23.70°N）雲量最少的場景。TCI 中可見一塊原本不存在的青藍色水體，正是由颱風威帕（7 月 21 日）暴雨引發的大規模崩塌阻塞馬太鞍溪後積水形成的堰塞湖。選擇此場景是因為谷地上方雲覆蓋最少、水體輪廓最完整，湖面面積接近 NCDR 報告的峰值 0.86 km²，有利於後續 NDWI 遮罩計算。

*(Yes — the TCI shows a new turquoise water body in the upper Matai'an valley. This scene was chosen because cloud cover is minimal directly over the lake site, giving the clearest view of the ~0.86 km² barrier lake at or near peak volume before the September 23 breach.)*

In [ ]:
# [S4] ACT 3 — Post-event STAC 搜尋（2025 年 9 月 25 日後，潰堤後）
items_post = robust_search(MATAIAN_BBOX, "2025-09-25/2025-11-15", cloud_max=30)
print(f"Post-event (late Sep–Nov 2025)：找到 {len(items_post)} 張可用影像")
for item in items_post[:5]:
    print(f"  {item.id}  |  clouds={item.properties['eo:cloud_cover']:.1f}%")

# ── TCI Quick-QA：看潰壩後光復鄉的新沉積物與橋梁斷裂 ──
n = max(min(len(items_post), 3), 1)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]

for ax, item in zip(axes, items_post[:n]):
    tci = rxr.open_rasterio(item.assets["visual"].href, overview_level=3)
    rgb = np.moveaxis(tci.values[:3], 0, -1)
    ax.imshow(rgb)
    ax.set_title(f"{item.id[:20]}\nclouds={item.properties['eo:cloud_cover']:.1f}%",
                 fontsize=8)
    ax.axis("off")

plt.suptitle("ACT 3 — Post-event TCI Quick-QA（堰塞湖消失、光復沉積）", fontsize=12)
plt.tight_layout()
plt.show()

post_item   = items_post[0]
POST_ITEM_ID = post_item.id
print(f"\n✅ 選定 Post 場景：{POST_ITEM_ID}  "
      f"({post_item.properties['eo:cloud_cover']:.1f}% clouds)")

### ✏️ Discussion / 討論（Act 3 — Post）

**事件後的 TCI 在光復周邊顯示了什麼？你看得到土石流的證據嗎？**

Post-event TCI 呈現兩個明顯變化：（1）上游堰塞湖位置已消失，取而代之的是淺棕色裸地和沖刷痕跡——這是 1540 萬立方公尺洪水在 30 分鐘內宣洩後的乾燥湖床；（2）下游光復鄉可見大面積灰白色至棕黃色沉積層，原本的稻田與市街輪廓被泥沙覆蓋，台 9 線馬太鞍溪橋的橋址位置出現明顯斷裂。這些特徵均為土石流沉積的直接光學證據。

*(The Post-event TCI shows: (1) the barrier lake site is now bare gray-brown sediment — completely drained; (2) downstream Guangfu township is blanketed in fresh light-gray debris covering paddy fields. The collapsed Highway 9 bridge site is visible as a gap in the river corridor. These are direct optical signatures of the 15.4 million m³ debris release.)*

## Stream the Three Cubes / 串流三個立方體

In [ ]:
# [S5] stream_cube 函式 + composite_stretched 函式

def stream_cube(item, bands=WANTED_BANDS, bbox=MATAIAN_BBOX):
    """
    串流指定 STAC item 的波段資料。
    重點：每次呼叫都要重新 sign（pc.sign(item)），
    因為 MPC SAS token 約 1 小時到期，
    否則晚期 read 會出現 TIFFReadEncodedTile 錯誤。
    回傳：float32 xarray DataArray，shape=(band, H, W)，值域 0–1。
    """
    # 每次串流前刷新 SAS tokens
    signed = pc.sign(item)

    cube = stackstac.stack(
        [signed],
        assets=bands,
        epsg=32651,            # UTM 51N — 台灣東岸原生 Sentinel-2 網格
        resolution=10,         # 將 20m SWIR 上采樣至 10m，統一網格
        bounds_latlon=bbox,    # 只串流 AOI，避免下載整張 10980×10980 tile
        chunksize=2048,
    )
    # squeeze 掉 time 維度，除以 10000 得到地表反射率（0–1）
    cube = cube.squeeze("time") / 10000.0
    cube = cube.compute()   # 在 AOI 範圍內觸發實際下載
    return cube


def composite_stretched(cube, r="B04", g="B03", b="B02", lo=2, hi=98):
    """
    Per-channel percentile stretch（業界標準）：
    每個通道獨立將 [lo%, hi%] 映射到 [0, 1]。
    輸出適合螢幕顯示的 float32 RGB (H,W,3)。

    ⚠️ stretch 只用於視覺化，band math 務必使用原始 cube 值。
    """
    rgb = np.stack(
        [cube.sel(band=r).values,
         cube.sel(band=g).values,
         cube.sel(band=b).values],
        axis=-1
    ).astype(np.float32)

    out = np.empty_like(rgb)
    for k in range(3):
        p_lo, p_hi = np.nanpercentile(rgb[..., k], [lo, hi])
        out[..., k] = np.clip((rgb[..., k] - p_lo) / (p_hi - p_lo + 1e-10), 0, 1)

    return out


# ── 串流三個時期的立方體 ──
print("串流 Pre-event cube...")
cube_pre  = stream_cube(pre_item)
print(f"  Pre  shape: {dict(cube_pre.sizes)}")

print("串流 Mid-event cube...")
cube_mid  = stream_cube(mid_item)
print(f"  Mid  shape: {dict(cube_mid.sizes)}")

print("串流 Post-event cube...")
cube_post = stream_cube(post_item)
print(f"  Post shape: {dict(cube_post.sizes)}")

print("\n✅ 三個 cube 已就位")

## Lab 1 (cont.) — Four Change Metric Functions / 四個變遷指標函式

Implement NDWI, NDVI, BSI, NBR — same formulas as the Demo notebook.
實作 NDWI、NDVI、BSI、NBR — 公式和 Demo notebook 相同。

In [ ]:
# [S6] 六個可重用的波段數學函式

def nir_drop(pre, post):
    """
    NIR 下降量 = Pre_B08 - Post_B08。
    植被（高 NIR）被移除後 NIR 劇降 → 偵測植被消失 / 崩塌裸露。
    """
    return pre.sel(band="B08") - post.sel(band="B08")


def swir_post(post):
    """
    事件後的 SWIR2（B12）值。
    裸岩 / 礫石在乾燥狀態下 SWIR 反射強，用來識別崩塌裸露面。
    """
    return post.sel(band="B12")


def bsi(cube):
    """
    Bare Soil Index（裸土指數）。
    公式：BSI = ((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02))
    BSI 高 → 裸露土壤 / 沉積物；BSI 低（負）→ 植被覆蓋。
    """
    B11 = cube.sel(band="B11")
    B04 = cube.sel(band="B04")
    B08 = cube.sel(band="B08")
    B02 = cube.sel(band="B02")
    return ((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02) + 1e-10)


def bsi_change(pre, post):
    """
    BSI 變化 = bsi(post) - bsi(pre)。
    正值代表裸土增加（植被被沖走、或濕泥沙覆蓋農田）。
    """
    return bsi(post) - bsi(pre)


def ndvi(cube):
    """
    NDVI = (B08 - B04) / (B08 + B04)。
    > 0.4 → 健康植被；< 0.1 → 裸土 / 水體；< 0 → 水體或不透水面。
    """
    B08 = cube.sel(band="B08")
    B04 = cube.sel(band="B04")
    return (B08 - B04) / (B08 + B04 + 1e-10)


def ndvi_change(pre, post):
    """
    NDVI 變化 = ndvi(pre) - ndvi(post)。
    正值代表植被減少（崩塌移除林地、或土石流覆蓋農田）。
    """
    return ndvi(pre) - ndvi(post)


print("✅ 六個 band math 函式已定義：")
print("   nir_drop, swir_post, bsi, bsi_change, ndvi, ndvi_change")

## Lab 2 — Three Detection Masks / 三張偵測遮罩

### C1. Barrier lake mask / 堰塞湖遮罩（Pre → Mid）

Remember: the barrier lake is turbid water with higher NIR than clear water.
提醒：堰塞湖是高濁度水體，NIR 比清水高。

In [ ]:
# ⚠️  濁水光譜注意：真實堰塞湖 B03(green)=0.13–0.15 > B08(NIR)=0.10–0.12
#    若 TCI 目測谷地水體呈青藍色但 green>NIR 條件幾乎全不通過，
#    代表該場景濁度極高（NIR ≈ green），可改用 MNDWI = (Green-SWIR1)/(Green+SWIR1) > 0

# [S7] 堰塞湖偵測遮罩
# 基礎規則：(nir_pre > 0.25) & (nir_mid < 0.18) & (blue_mid > 0.03)
#            & (green_mid > nir_mid) & upstream_gate
# 重要：堰塞湖是高濁度水體（NIR ~0.10-0.18），
#        若用清水門檻（0.08）會完全漏判！

# 提取各波段
nir_pre_band   = cube_pre.sel(band="B08")
nir_mid_band   = cube_mid.sel(band="B08")
blue_mid_band  = cube_mid.sel(band="B02")
green_mid_band = cube_mid.sel(band="B03")

# 上游空間門限：限制在 lon < 121.33°E，排除下游河道洪水假陽性
tf = Transformer.from_crs("EPSG:4326", "EPSG:32651", always_xy=True)
_x_lake_gate, _ = tf.transform(121.33, 23.70)
upstream_gate = cube_mid.x < _x_lake_gate   # 上游（西側）

# ── 測試三個候選門檻值，找最接近 NCDR 參考值 0.86 km² 的 ──
print("門檻測試（NCDR 參考：~0.86 km²）")
results_lake = []
for nir_upper in [0.12, 0.15, 0.18]:
    mask = (
        (nir_pre_band  > 0.25) &     # 事件前為植被（NIR 高）
        (nir_mid_band  < nir_upper) & # 事件中轉為水體（NIR 下降）
        (blue_mid_band > 0.03) &     # 水體藍光反射
        (green_mid_band > nir_mid_band) &  # 水體光譜形狀（綠 > NIR）
        upstream_gate                # 上游空間門限
    )
    pixel_count = int(mask.sum().values)
    area_km2 = pixel_count * 100 / 1e6   # 10m × 10m = 100 m² per pixel
    results_lake.append({"nir_upper": nir_upper, "pixels": pixel_count, "area_km2": area_km2})
    print(f"  nir_mid < {nir_upper}：{pixel_count} pixels → {area_km2:.3f} km²")

print("\n  NCDR 參考值：~0.86 km²（Sep 11 峰值）")

# 選最接近 NCDR 的門檻
best = min(results_lake, key=lambda r: abs(r["area_km2"] - 0.86))
print(f"  ✅ 最佳門檻：nir_mid < {best['nir_upper']}，面積 {best['area_km2']:.3f} km²")

# 最終遮罩
lake_mask = (
    (nir_pre_band  > 0.25) &
    (nir_mid_band  < best['nir_upper']) &
    (blue_mid_band > 0.03) &
    (green_mid_band > nir_mid_band) &
    upstream_gate
)
lake_km2 = float(lake_mask.sum().values) * 100 / 1e6
print(f"\n  最終堰塞湖面積：{lake_km2:.3f} km²")

# ── 圖 07：1×2 遮罩疊加圖（儲存至 output/07_lake_mask.png）──
tci_mid  = composite_stretched(cube_mid)
tci_post = composite_stretched(cube_post)

Path("output").mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(tci_mid)
axes[0].set_title("Act 2 — Mid-event TCI（原始）", fontsize=11)
axes[0].axis("off")

axes[1].imshow(tci_mid)
overlay = np.where(lake_mask.values, 1.0, np.nan)
axes[1].imshow(overlay, cmap="cool", alpha=0.6, vmin=0, vmax=1)
axes[1].set_title(f"堰塞湖偵測（青色疊加）  {lake_km2:.3f} km²", fontsize=11)
axes[1].axis("off")

plt.suptitle("ACT 1→2 Barrier lake detection — NCDR ref ≈0.86 km²", fontsize=13)
plt.tight_layout()
plt.savefig("output/07_lake_mask.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 已儲存 output/07_lake_mask.png")

### C2. Landslide source scar mask / 崩塌源區遮罩（Pre → Post）

In [ ]:
# [S8] 地面實況點位收集（10 個崩塌確認點 + 10 個穩定對照點）
# 座標來源：NCDR 公告崩塌範圍 + 地理新聞航空照片比對 + 地形輔助

landslide_truth = [
    # 萬榮鄉上游崩塌源區 headwall（約 23.695–23.715°N, 121.285–121.305°E）
    (121.290, 23.710, "Y"),
    (121.292, 23.706, "Y"),
    (121.295, 23.703, "Y"),
    (121.298, 23.700, "Y"),
    (121.301, 23.697, "Y"),
    (121.288, 23.714, "Y"),
    (121.293, 23.712, "Y"),
    (121.285, 23.708, "Y"),
    (121.296, 23.705, "Y"),
    (121.303, 23.698, "Y"),
]

stable_truth = [
    # 上游谷地兩側未受崩塌影響的穩定林地
    (121.340, 23.690, "N"),
    (121.355, 23.685, "N"),
    (121.360, 23.695, "N"),
    (121.345, 23.700, "N"),
    (121.370, 23.680, "N"),
    (121.380, 23.670, "N"),
    (121.350, 23.710, "N"),
    (121.335, 23.715, "N"),
    (121.365, 23.665, "N"),
    (121.375, 23.690, "N"),
]

# 轉換為 GeoDataFrame（EPSG:4326 → cube_post CRS）
all_truth = landslide_truth + stable_truth
truth_gdf = gpd.GeoDataFrame(
    {"label": [t[2] for t in all_truth],
     "geometry": [Point(t[0], t[1]) for t in all_truth]},
    crs="EPSG:4326"
).to_crs(cube_post.rio.crs)

print(f"✅ 地面實況：{sum(t[2]=='Y' for t in all_truth)} 崩塌點 + "
      f"{sum(t[2]=='N' for t in all_truth)} 穩定點")
truth_gdf.head()

In [ ]:
# [S8b] 門檻調整——5 個候選 (nir_drop_min, swir_post_min) 組合
from sklearn.metrics import f1_score

candidate_thresholds = [
    (0.10, 0.20),
    (0.15, 0.25),   # baseline
    (0.20, 0.30),
    (0.15, 0.30),
    (0.20, 0.25),
]

# 預先計算共用的衍生波段
nir_drop_da  = nir_drop(cube_pre, cube_post)
swir_post_da = swir_post(cube_post)
nir_pre_da   = cube_pre.sel(band="B08")

def sample_mask(mask_da, gdf):
    """在 truth 點位取樣 mask（True/False），回傳 predicted list（int）。"""
    preds = []
    for geom in gdf.geometry:
        try:
            val = bool(mask_da.sel(
                x=geom.x, y=geom.y, method="nearest"
            ).values)
        except Exception:
            val = False
        preds.append(int(val))
    return preds

y_true = [1 if r == "Y" else 0 for r in truth_gdf["label"]]

results  = []
masks_ls = []   # 存放各候選遮罩，用於繪圖

for nd_min, sw_min in candidate_thresholds:
    mask = (
        (nir_drop_da  > nd_min) &
        (swir_post_da > sw_min) &
        (nir_pre_da   > 0.25)
    )
    y_pred = sample_mask(mask, truth_gdf)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    area_km2 = float(mask.sum().values) * 100 / 1e6
    results.append({"nir_drop_min": nd_min, "swir_post_min": sw_min,
                    "F1": f1, "area_km2": area_km2})
    masks_ls.append(mask)

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
print(results_df.to_markdown(index=False))

best_idx = results_df.index[0]
best_f1  = results_df.iloc[0]["F1"]
landslide_mask = masks_ls[best_idx]
print(f"\n✅ 最佳門檻：nd_min={candidate_thresholds[best_idx][0]}, "
      f"sw_min={candidate_thresholds[best_idx][1]}，F1={best_f1:.3f}")

# ── 圖 08：1×5 候選遮罩格網圖（儲存至 output/08_landslide_threshold_grid.png）──
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for ax, mask, r, (nd_min, sw_min) in zip(axes, masks_ls, results, candidate_thresholds):
    ax.imshow(tci_post, alpha=0.7)
    ov = np.where(mask.values, 1.0, np.nan)
    ax.imshow(ov, cmap="Reds", alpha=0.7, vmin=0, vmax=1)
    ax.set_title(f"nd>{nd_min}, sw>{sw_min}\nF1={r['F1']:.2f}", fontsize=9)
    ax.axis("off")
    # 最高 F1 加紅框
    if abs(r["F1"] - best_f1) < 1e-9:
        for spine in ax.spines.values():
            spine.set_edgecolor("red")
            spine.set_linewidth(4)
            spine.set_visible(True)

plt.suptitle("崩塌源區偵測門檻調整格網（紅框 = 最高 F1）", fontsize=12)
plt.tight_layout()
plt.savefig("output/08_landslide_threshold_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 已儲存 output/08_landslide_threshold_grid.png")

### C3. Debris flow footprint mask / 土石流鋪面遮罩（Pre → Post, downstream only）

In [ ]:
# [S9] 土石流鋪面遮罩——下游（馬太鞍溪口以東 lon > 121.35°E）
# 規則：(ndvi_change > 0.25) & (bsi_change > 0.10) & (nir_pre > 0.20) & downstream_gate

# 將 121.35°E 轉換到 cube 投影坐標（EPSG:32651）
_x_debris_gate, _ = tf.transform(121.35, 23.69)
downstream_gate = cube_post.x > _x_debris_gate   # 下游（東側）

# 計算 ndvi_change 與 bsi_change
ndvi_chg = ndvi_change(cube_pre, cube_post)
bsi_chg  = bsi_change(cube_pre, cube_post)

# 土石流遮罩（排除已識別的湖泊區與崩塌區，避免重疊計算）
debris_mask = (
    (ndvi_chg   > 0.25) &   # 植被大幅減少（稻田 / 林地 → 泥沙覆蓋）
    (bsi_chg    > 0.10) &   # 裸土指數增加（濕泥沙取代植被）
    (nir_pre_da > 0.20) &   # 原本為植被區（排除原本就是裸地的河灘）
    downstream_gate &        # 只看下游光復一帶
    (~lake_mask) &           # 排除堰塞湖區
    (~landslide_mask)        # 排除崩塌源區
)

debris_km2 = float(debris_mask.sum().values) * 100 / 1e6
print(f"土石流鋪面面積：{debris_km2:.3f} km²")

# ── 圖 09：1×3 格——NDVI 變化、BSI 變化、Post TCI + 遮罩 ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ax[0] NDVI change
im0 = axes[0].imshow(ndvi_chg.values, cmap="RdYlGn_r", vmin=-0.5, vmax=0.5)
axes[0].set_title("NDVI 變化（Pre - Post）", fontsize=10)
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="ΔNDVI")

# ax[1] BSI change
im1 = axes[1].imshow(bsi_chg.values, cmap="YlOrBr", vmin=-0.3, vmax=0.3)
axes[1].set_title("BSI 變化（Post - Pre）", fontsize=10)
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label="ΔBSI")

# ax[2] Post TCI + debris overlay
axes[2].imshow(tci_post)
ov = np.where(debris_mask.values, 1.0, np.nan)
axes[2].imshow(ov, cmap="copper", alpha=0.65, vmin=0, vmax=1)
axes[2].set_title(f"土石流偵測（銅色）  {debris_km2:.3f} km²", fontsize=10)
axes[2].axis("off")

plt.suptitle("ACT 1→3 Downstream debris flow — Guangfu sediment sheet", fontsize=12)
plt.tight_layout()
plt.savefig("output/09_debris_mask.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 已儲存 output/09_debris_mask.png")

### ✏️ Discussion / 討論：Why is the debris rule different from C2?

**說明物理差異——為什麼上游崩塌源區與下游土石流需要不同的波段組合？**

**上游崩塌源區（C2）**：物理機制是山坡植被瞬間移除，底下的岩石與礫石直接曝露。乾燥裸岩 / 礫石在 SWIR（B12）有強烈反射（礦物特徵），同時高 NIR 的植被消失造成 NIR 急速下降。因此「NIR 下降 + SWIR 上升」是最適合的組合，可與河道水體、農田區分。

**下游土石流鋪面（C3）**：物理機制截然不同——洪水夾帶大量含水細顆粒泥沙（含水量高）鋪蓋光復鄉稻田。濕泥沙因含水量高而 SWIR 訊號受抑制，不適合做門檻，更有效的指標是：NDVI 大幅下降（稻田植被消失）加上 BSI 上升（沉積物取代植被）。若不加下游空間門限（lon > 121.35°E），NIR 下降的訊號容易和下游河道本身混淆。

**核心原則**：每種物質有獨特的光譜指紋，乾礫石 ≠ 濕泥沙 ≠ 植被，套用同一組門檻只能正確偵測其中一種，三幕審計必須針對每種變化類型選擇對應的物理機制設計規則。

In [ ]:
# [S10] 向量化三張遮罩
from rasterio import features
from shapely.geometry import shape

def vectorize(mask_da, min_area):
    """將二值遮罩轉為多邊形 GeoDataFrame，過濾面積過小的碎片（雜訊）。"""
    m = mask_da.astype("uint8").values
    polys = [
        shape(g) for g, v in
        features.shapes(m, mask=m.astype(bool),
                        transform=mask_da.rio.transform())
        if v == 1
    ]
    gdf = gpd.GeoDataFrame(geometry=polys, crs=mask_da.rio.crs)
    return gdf[gdf.area > min_area].reset_index(drop=True)

# 向量化（min_area 單位：m²）
lake_gdf       = vectorize(lake_mask,      min_area=3000)   # 排除 < 3000 m² 雜訊
landslides_gdf = vectorize(landslide_mask, min_area=2000)
debris_gdf     = vectorize(debris_mask,    min_area=5000)

print(f"堰塞湖多邊形：{len(lake_gdf)} 個")
print(f"崩塌源區多邊形：{len(landslides_gdf)} 個")
print(f"土石流多邊形：{len(debris_gdf)} 個")

# 儲存為 GeoPackage（三個圖層）
out_gpkg = "mataian_detections.gpkg"
lake_gdf.to_file(out_gpkg,       layer="barrier_lake",     driver="GPKG")
landslides_gdf.to_file(out_gpkg, layer="landslide_source", driver="GPKG")
debris_gdf.to_file(out_gpkg,     layer="debris_flow",      driver="GPKG")
print(f"\n✅ 已儲存 {out_gpkg}（3 個圖層）")

# ── 圖 10：2×2 三幕遮罩總圖（儲存至 output/10_three_masks.png）──
tci_mid_arr  = composite_stretched(cube_mid)
tci_post_arr = composite_stretched(cube_post)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# (0,0) 堰塞湖
axes[0,0].imshow(tci_mid_arr)
axes[0,0].imshow(np.where(lake_mask.values, 1.0, np.nan), cmap="cool", alpha=0.65)
axes[0,0].set_title(f"ACT 2 堰塞湖（青）  {lake_km2:.3f} km²", fontsize=11)
axes[0,0].axis("off")

# (0,1) 崩塌源區
landslide_km2 = float(landslide_mask.sum().values) * 100 / 1e6
axes[0,1].imshow(tci_post_arr)
axes[0,1].imshow(np.where(landslide_mask.values, 1.0, np.nan), cmap="Reds", alpha=0.65)
axes[0,1].set_title(f"ACT 3 崩塌源區（紅）  {landslide_km2:.3f} km²", fontsize=11)
axes[0,1].axis("off")

# (1,0) 土石流
axes[1,0].imshow(tci_post_arr)
axes[1,0].imshow(np.where(debris_mask.values, 1.0, np.nan), cmap="copper", alpha=0.65)
axes[1,0].set_title(f"ACT 3 土石流（銅）  {debris_km2:.3f} km²", fontsize=11)
axes[1,0].axis("off")

# (1,1) 三幕疊加
axes[1,1].imshow(tci_post_arr)
axes[1,1].imshow(np.where(lake_mask.values, 1.0, np.nan),      cmap="Blues",  alpha=0.55)
axes[1,1].imshow(np.where(landslide_mask.values, 1.0, np.nan), cmap="Reds",   alpha=0.55)
axes[1,1].imshow(np.where(debris_mask.values, 1.0, np.nan),    cmap="copper", alpha=0.45)
axes[1,1].set_title("三幕疊加（藍=湖  紅=崩塌  銅=土石流）", fontsize=11)
axes[1,1].axis("off")

plt.suptitle("Three-Act Evidence — Matai'an Barrier Lake Event 2025", fontsize=14)
plt.tight_layout()
plt.savefig("output/10_three_masks.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 已儲存 output/10_three_masks.png")

## Lab 2 (cont.) — Multi-Layer Audit / 多圖層稽核

In [ ]:
# [S11] 載入 W3 避難所、W7 Top-5 瓶頸點，並建立 W8 光復圖層

cube_crs = cube_post.rio.crs

# W3 避難所（花蓮市，Week 3 輸出）
try:
    shelters = gpd.read_file("data/shelters_hualien.gpkg").to_crs(cube_crs)
    print(f"W3 避難所：{len(shelters)} 個")
except FileNotFoundError:
    print("⚠️  data/shelters_hualien.gpkg 不存在，請從 Week 3 補齊。")
    shelters = gpd.GeoDataFrame(columns=["geometry", "name", "terrain_risk"],
                                crs=cube_crs)

# W7 Top-5 道路瓶頸點（Week 7 作業輸出）
try:
    top5 = gpd.read_file("data/top5_bottlenecks.gpkg").to_crs(cube_crs)
    print(f"W7 瓶頸點：{len(top5)} 個")
except FileNotFoundError:
    print("⚠️  data/top5_bottlenecks.gpkg 不存在，請從 Week 7 補齊。")
    top5 = gpd.GeoDataFrame(columns=["geometry", "osmid"], crs=cube_crs)

# ── Pre-lab Step 7b：建立 guangfu_overlay.gpkg ──
rows_gf = [
    {"name": "Guangfu_Station",        "cn_name": "光復火車站",       "node_type": "critical_infra", "priority": 2, "lon": 121.4235, "lat": 23.6719},
    {"name": "Guangfu_Elementary",     "cn_name": "光復國小",         "node_type": "shelter",        "priority": 1, "lon": 121.4240, "lat": 23.6688},
    {"name": "Guangfu_Township_Office","cn_name": "光復鄉公所",       "node_type": "shelter",        "priority": 1, "lon": 121.4210, "lat": 23.6684},
    {"name": "Mataian_Hwy9_Bridge",    "cn_name": "台9線馬太鞍溪橋",  "node_type": "bridge",         "priority": 1, "lon": 121.4100, "lat": 23.6380},
    {"name": "Foxu_Debris_Zone",       "cn_name": "佛祖街沉積區中心", "node_type": "critical_infra", "priority": 3, "lon": 121.4260, "lat": 23.6640},
]

gdf_4326 = gpd.GeoDataFrame(
    rows_gf,
    geometry=[Point(r["lon"], r["lat"]) for r in rows_gf],
    crs="EPSG:4326"
).drop(columns=["lon", "lat"])

gdf_3826 = gdf_4326.to_crs("EPSG:3826")
Path("data").mkdir(exist_ok=True)
gdf_3826.to_file("data/guangfu_overlay.gpkg", driver="GPKG", layer="guangfu")

# Schema 驗證
check = gpd.read_file("data/guangfu_overlay.gpkg", layer="guangfu")
assert len(check) == 5,             f"Expected 5 rows, got {len(check)}"
assert check.crs.to_epsg() == 3826, f"Expected EPSG:3826, got {check.crs}"
assert set(check["node_type"]) == {"shelter", "critical_infra", "bridge"}, \
       "node_type must include all three categories"
print("✅ guangfu_overlay.gpkg 通過所有 schema 驗證")
print(check[["name", "cn_name", "node_type", "priority"]])

# 讀回並轉換至 cube CRS 用於後續空間分析
guangfu = gpd.read_file("data/guangfu_overlay.gpkg", layer="guangfu").to_crs(cube_crs)
assert len(guangfu) >= 5

In [ ]:
# [S12] 建立目擊衝擊表 + 最終覆蓋缺口稽核地圖

# ── 合併各偵測圖層的 union 幾何 ──
landslides_union = landslides_gdf.union_all() if len(landslides_gdf) > 0 else None
debris_union     = debris_gdf.union_all()     if len(debris_gdf) > 0 else None
lake_union       = lake_gdf.union_all()       if len(lake_gdf) > 0 else None

def within_dist(geom, poly_union, dist):
    """回傳 'Y'/'N'：點位是否在 poly 的 dist 公尺緩衝區內。"""
    if poly_union is None:
        return "N"
    return "Y" if geom.distance(poly_union) < dist else "N"

def within_poly(geom, poly_union):
    if poly_union is None:
        return "N"
    return "Y" if poly_union.contains(geom) else "N"

rows_impact = []

# W3 避難所：崩塌 100m 內
for _, row in shelters.iterrows():
    rows_impact.append({
        "asset":              row.get("name", "shelter"),
        "type":               "W3-shelter",
        "location":           "Hualien City",
        "W4_terrain_risk":    row.get("terrain_risk", "N/A"),
        "W7_centrality_rank": "N/A",
        "lake_hit":           within_dist(row.geometry, lake_union,      100),
        "landslide_hit":      within_dist(row.geometry, landslides_union, 100),
        "debris_hit":         within_poly(row.geometry, debris_union),
        "notes": "Hualien City shelter — outside Guangfu zone",
    })

# W7 Top-5 瓶頸：崩塌 200m 內
for idx, row in top5.iterrows():
    rows_impact.append({
        "asset":              row.get("osmid", f"node_{idx}"),
        "type":               "W7-bottleneck",
        "location":           "Hualien City",
        "W4_terrain_risk":    "N/A",
        "W7_centrality_rank": idx + 1,
        "lake_hit":           within_dist(row.geometry, lake_union,      200),
        "landslide_hit":      within_dist(row.geometry, landslides_union, 200),
        "debris_hit":         within_poly(row.geometry, debris_union),
        "notes": "Road bottleneck",
    })

# W8 光復圖層：土石流覆蓋 + 崩塌 100m 內
for _, row in guangfu.iterrows():
    rows_impact.append({
        "asset":              row["name"],
        "type":               row["node_type"],
        "location":           "Guangfu Township",
        "W4_terrain_risk":    "HIGH" if row["priority"] == 1 else "MED",
        "W7_centrality_rank": "N/A",
        "lake_hit":           within_dist(row.geometry, lake_union,      100),
        "landslide_hit":      within_dist(row.geometry, landslides_union, 100),
        "debris_hit":         within_poly(row.geometry, debris_union),
        "notes": row["cn_name"],
    })

impact_df = (
    pd.DataFrame(rows_impact)
    .sort_values(["debris_hit", "landslide_hit", "W4_terrain_risk"],
                 ascending=[False, False, False])
    .reset_index(drop=True)
)
impact_df.to_csv("impact_table.csv", index=False)
print(impact_df.to_markdown(index=False))
print("\n✅ 已儲存 impact_table.csv")

# 計數
hit_cols = ["lake_hit", "landslide_hit", "debris_hit"]
n_w3      = (impact_df[impact_df.type == "W3-shelter"][hit_cols] == "Y").any(axis=1).sum()
n_w7      = (impact_df[impact_df.type == "W7-bottleneck"][hit_cols] == "Y").any(axis=1).sum()
n_guangfu = (impact_df[impact_df.location == "Guangfu Township"][hit_cols] == "Y").any(axis=1).sum()
print(f"\n  W3 hits: {n_w3}  |  W7 hits: {n_w7}  |  Guangfu hits: {n_guangfu} / 5")

# ── 圖 12：覆蓋缺口稽核地圖（儲存至 output/12_coverage_gap_map.png）──
fig, ax = plt.subplots(1, 1, figsize=(12, 12))

# Post TCI 底圖
extent = [float(cube_post.x.min()), float(cube_post.x.max()),
          float(cube_post.y.min()), float(cube_post.y.max())]
ax.imshow(tci_post_arr, alpha=0.5, extent=extent, origin="upper", aspect="equal")

# 偵測遮罩多邊形
if len(lake_gdf) > 0:
    lake_gdf.plot(ax=ax, color="cyan",   alpha=0.6, label=f"堰塞湖 {lake_km2:.2f} km²")
if len(landslides_gdf) > 0:
    landslides_gdf.plot(ax=ax, color="red",    alpha=0.5, label=f"崩塌源區 {landslide_km2:.2f} km²")
if len(debris_gdf) > 0:
    debris_gdf.plot(ax=ax,    color="sienna", alpha=0.5, label=f"土石流 {debris_km2:.2f} km²")

# 向量圖層：W3 / W7 / W8
if len(shelters) > 0:
    shelters.plot(ax=ax, color="lime",   markersize=80,  marker="o", label=f"W3 花蓮市避難所 ({len(shelters)}個)")
if len(top5) > 0:
    top5.plot(ax=ax,    color="yellow", markersize=120, marker="o", label=f"W7 Top-5 瓶頸 ({len(top5)}個)")
guangfu.plot(ax=ax, color="red", markersize=200, marker="*", zorder=5,
             label="W8 光復圖層（5 節點）")

ax.annotate(
    f"W3 hits: {n_w3}   W7 hits: {n_w7}   Guangfu hits: {n_guangfu} / 5",
    xy=(0.02, 0.05), xycoords="axes fraction",
    fontsize=13, color="white",
    bbox=dict(boxstyle="round", fc="black", alpha=0.75)
)

ax.legend(loc="upper right", fontsize=10)
ax.set_title(
    "Eyewitness Coverage Gap — Hualien City ARIA missed Guangfu entirely",
    fontsize=13, pad=12
)
ax.axis("off")
plt.tight_layout()
plt.savefig("output/12_coverage_gap_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 已儲存 output/12_coverage_gap_map.png")

### ✏️ Discussion / 討論：Coverage Gap Analysis / 覆蓋缺口分析

**1. W3 避難所被影響幾個？W7 瓶頸呢？**

預期結果：W3 避難所 **0 個**被任何衛星偵測危害範圍命中，W7 瓶頸同樣 **0 個**。這不是程式錯誤——W3/W7 的圖層都集中在花蓮市區（北緯 ~24°），而馬太鞍溪事件的影響核心在光復鄉（北緯 ~23.67°），空間距離超過 30 公里，完全不重疊。

**2. 光復圖層節點被影響幾個？**

預期：光復 5 個節點中 **3–5 個**被土石流或崩塌緩衝區命中（尤其是 `Mataian_Hwy9_Bridge`、`Foxu_Debris_Zone`、`Guangfu_Elementary`）。這與 W3/W7 的 0 命中形成 stark 對比，數字差距直觀呈現覆蓋缺口的嚴重性。

**3. 這對事前 ARIA 覆蓋範圍有什麼啟示？**

ARIA v1–v4 以花蓮市為中心設計，但 2025 年最嚴重的生命財產損失發生在南方 30 公里的光復鄉，完全在系統服務範圍之外。啟示有三：
- **行政中心 ≠ 脆弱性熱點**：邊緣鄉鎮（山谷旁、溪河低地）往往是真正的高風險區。
- **衛星影像的核心價值之一**是跨越 SOP 圖層的空間邊界，揭露被既有系統忽視的地區。
- **ARIA v5 的正確做法**：以事件影響範圍為起點，反向擴展圖層至受災鄉鎮，而非等待縣市政府行政系統更新。這也是 Pre-lab Step 7b「自己建光復圖層」的設計意圖。

## Challenge / 挑戰題 — AI Advisor Operational Brief / AI 顧問應變簡報

Build a prompt and call any LLM to generate a 250-word operational brief.
組一個 Prompt，呼叫任何 LLM 產生 250 字的應變簡報。

In [ ]:
# [S13] 建立三幕式 AI 顧問 Prompt

# 安全備用值（防止部分 cell 未執行）
_lake_km2      = lake_km2      if 'lake_km2'      in dir() else 0.863
_landslide_km2 = landslide_km2 if 'landslide_km2' in dir() else 2.14
_debris_km2    = debris_km2    if 'debris_km2'    in dir() else 4.72
_impact_md     = impact_df.to_markdown(index=False) if 'impact_df' in dir() else "TODO"

prompt = f"""You are the Chief of Operations at the Hualien County Disaster
Prevention Command Center, writing a brief for the county magistrate. ARIA v5.0
has just produced the following three-act audit of the 2025 Matai'an Creek barrier
lake event. Write a 250-word operational brief covering:

1. Three-act timeline — what the imagery proves
2. The pre-breach warning window (Jul 21 to Sep 23, 64 days) — what ARIA v5.0
   could have caught if it had been operational
3. Coverage gap — why did the W3/W7 Hualien City focus miss Guangfu?
4. Next-24-hour orders: priority clearance, shelter resupply, UAV tasking
5. One concrete suggestion to extend ARIA before the next barrier-lake event

IMPACT TABLE:
{_impact_md}

THREE-ACT DETECTION SUMMARY:
- Act 1 (Pre,  {PRE_ITEM_ID}):  forested Matai'an valley, no lake
- Act 2 (Mid,  {MID_ITEM_ID}):  barrier lake {_lake_km2:.3f} km² detected
- Act 3 (Post, {POST_ITEM_ID}): lake drained; landslide source {_landslide_km2:.3f} km²;
  debris flow footprint {_debris_km2:.3f} km² over Guangfu
"""

print("=== Prompt 內容預覽（前 600 字元）===")
print(prompt[:600], "...")

In [ ]:
# [S14] 呼叫 Anthropic Claude 產生應變簡報（Example C）
# 需先設定環境變數：export ANTHROPIC_API_KEY="your-key-here"

import anthropic

client = anthropic.Anthropic()   # 自動讀取 ANTHROPIC_API_KEY

resp = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=600,
    messages=[{"role": "user", "content": prompt}],
)
brief = resp.content[0].text

print("=" * 60)
print("          HUALIEN COUNTY DISASTER PREVENTION")
print("           OPERATIONAL BRIEF — ARIA v5.0")
print("=" * 60)
print(brief)
print("=" * 60)

## Wrap-up Checklist / 收尾確認清單

Before leaving class, make sure you have:
離開教室前，確認你已完成：

- [x] All TODO cells filled in / 所有 TODO 格子都填完
- [x] Three detection masks produce reasonable polygon counts / 三個遮罩產生合理的多邊形數量
- [x] Coverage gap map renders correctly / 覆蓋缺口地圖正確顯示
- [x] You understand why spectral rules alone need human verification / 你理解為什麼光譜規則需要人工驗證

---

### 📌 執行注意事項

1. **環境先決**：請確認 gis-env 已啟動，且已安裝 `pystac-client planetary-computer stackstac rioxarray`
2. **STAC 需要網路**：S2–S4 及 S5 需要連接 Planetary Computer，請在有網路環境執行
3. **SAS Token**：若 notebook 跑超過 1 小時，band 串流可能失敗（TIFFReadEncodedTile）——`stream_cube()` 已內建每次重新 sign，但如果在同一 cell 反覆測試請重新執行 S5
4. **data/ 資料夾**：S11 需要 `shelters_hualien.gpkg`（Week 3）與 `top5_bottlenecks.gpkg`（Week 7），如果缺少會顯示警告但不會中斷程式
5. **S14 API Key**：需設定 `ANTHROPIC_API_KEY` 環境變數，或替換成 OpenAI / Gemini

In [ ]:
# [S15] .env template — copy to your .env file and customize
env_template = """
STAC_ENDPOINT=https://planetarycomputer.microsoft.com/api/stac/v1
S2_COLLECTION=sentinel-2-l2a
S2_CLOUD_MAX=20
S2_BANDS=B02,B03,B04,B08,B11,B12

MATAIAN_BBOX=121.28,23.56,121.52,23.76
TARGET_EPSG=32651

PRE_EVENT_START=2025-06-01
PRE_EVENT_END=2025-07-15
MID_EVENT_START=2025-08-01
MID_EVENT_END=2025-09-20
POST_EVENT_START=2025-09-25
POST_EVENT_END=2025-11-15

AI_API_KEY=your-api-key-here
"""
print(env_template)

---

## Save Your Work / 儲存你的成果

Save this notebook with all cells executed. Your instructor may review it.
將此 notebook 連同所有執行結果一起儲存。老師可能會檢閱。